# T6.2 LLM extraction on Colab Pro+

Rerouted from OSC (DECISIONS 2026-07-04). Same vLLM + Outlines + YaRN engine, on a Colab GPU.
Read `cloud/colab/README.md` + `RUNBOOK.md` first. Prereqs: Pro+ with a **GPU runtime**
(Runtime → Change runtime type → A100/L4), the repo on Drive at the `REPO` path below, and the
two gitignored `chunks.parquet` staged under its `data/`.

Reproducibility rule: a model's κ-audit must score the exact weights that produced its corpus
features — so `--audit-sample` and the corpus run use the same model + engine in the same session.

In [ ]:
# 1. Mount Drive (persistent store: code + the data/ outputs survive VM disconnects)
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# 2. cd into the repo on Drive + install (idempotent; safe to re-run after a reconnect)
REPO = "/content/drive/MyDrive/ecvol/Earnings_call_project-main"  # <-- edit if yours differs
%cd $REPO
!bash cloud/colab/setup.sh

In [ ]:
# 3. Config the run
MODEL = "Qwen/Qwen2.5-7B-Instruct"  # panel below
# panel: Qwen/Qwen2.5-32B-Instruct-AWQ (40GB), meta-llama/Llama-3.1-8B-Instruct
CTX = "--max-model-len 65536 --yarn"  # >32k policy: extend (must match audit+corpus)
SLUG = MODEL.replace("/", "__").replace(":", "_")
print(MODEL, "->", f"data/*/llm_features__{SLUG}.parquet")

In [ ]:
# 4. Smoke test — 3 calls, validates the vLLM/YaRN path cheaply
!ecvol featurize llm --dataset fincall --model-id $MODEL \
    --engine vllm $CTX --limit 3 --root data

In [ ]:
# 5. κ-GATE — extract the 50 audit calls (same engine), then score; corpus BLOCKED until PASS
!ecvol featurize llm --dataset fincall --model-id $MODEL \
    --engine vllm $CTX --audit-sample --root data
!ecvol llm-kappa --sheet data/coverage/fincall_llm_labels_rater1.csv \
    --features data/fincall/llm_features__$SLUG.parquet

In [ ]:
# 6. Full corpus — ONLY if the gate passed. Resumes over audit calls; re-run after a disconnect.
!ecvol featurize llm --dataset fincall --model-id $MODEL \
    --engine vllm $CTX --root data
!ecvol featurize llm --dataset maec --model-id $MODEL \
    --engine vllm $CTX --root data